# 04 — Pattern Balance Model

Този notebook разглежда структурния баланс на тиражите: odd/even, low/high, sum, spread и consecutive pairs.


In [ ]:
from pathlib import Path
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"


def read_csv_safe(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def number_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in ["n1", "n2", "n3", "n4", "n5", "n6"] if col in df.columns]


def parse_numbers_text(value) -> list[int]:
    if pd.isna(value):
        return []
    parts = str(value).replace(";", ",").replace("|", ",").split(",")
    nums = []
    for part in parts:
        part = part.strip()
        if part.isdigit():
            nums.append(int(part))
    return nums


def extract_ticket_numbers(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    if len(number_columns(df)) == 6:
        return df.copy()
    text_col = next((c for c in df.columns if "numbers" in c.lower() or "числа" in c.lower()), None)
    if text_col is None:
        return df.copy()
    out = df.copy()
    parsed = out[text_col].apply(parse_numbers_text)
    for i in range(6):
        out[f"n{i+1}"] = parsed.apply(lambda nums: nums[i] if len(nums) > i else np.nan)
    return out


def show_latest_draw(df: pd.DataFrame) -> None:
    if df.empty:
        print("No draw data loaded.")
        return
    latest = df.tail(1).iloc[0]
    nums = [int(latest[col]) for col in number_columns(df)]
    print("Rows:", len(df))
    print("Latest date:", latest.get("date", "n/a"))
    print("Draw number:", latest.get("draw_number", latest.get("draw_no", "n/a")))
    print("Numbers:", nums)


def file_status(paths):
    rows = []
    for path in paths:
        path = Path(path)
        rows.append({
            "file": str(path.relative_to(PROJECT_ROOT)) if str(path).startswith(str(PROJECT_ROOT)) else str(path),
            "exists": path.exists(),
            "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else 0,
        })
    return pd.DataFrame(rows)


draws = read_csv_safe(DATA_DIR / "historical_draws.csv")
normalized = read_csv_safe(DATA_DIR / "v40_normalized_draw_events.csv")
canonical = read_csv_safe(DATA_DIR / "v41_canonical_draw_events.csv")
show_latest_draw(draws)


## Pattern features


In [ ]:
num_cols = number_columns(draws)
patterns = draws.copy()
patterns["sum"] = patterns[num_cols].sum(axis=1)
patterns["odd_count"] = patterns[num_cols].apply(lambda row: sum(int(x) % 2 == 1 for x in row), axis=1)
patterns["even_count"] = 6 - patterns["odd_count"]
patterns["low_count"] = patterns[num_cols].apply(lambda row: sum(int(x) <= 24 for x in row), axis=1)
patterns["high_count"] = 6 - patterns["low_count"]
patterns["spread"] = patterns[num_cols].max(axis=1) - patterns[num_cols].min(axis=1)

def consecutive_pairs(row):
    nums = sorted(int(x) for x in row)
    return sum(1 for a, b in zip(nums, nums[1:]) if b - a == 1)
patterns["consecutive_pairs"] = patterns[num_cols].apply(consecutive_pairs, axis=1)
patterns[["date", "draw_number", "sum", "odd_count", "even_count", "low_count", "high_count", "spread", "consecutive_pairs"]].tail(10)


## Distributions


In [ ]:
odd_counts = patterns["odd_count"].value_counts().sort_index()
ax = odd_counts.plot(kind="bar", figsize=(8, 4), title="Odd count distribution")
ax.set_xlabel("Odd numbers in draw")
ax.set_ylabel("Draw count")
plt.show()
low_counts = patterns["low_count"].value_counts().sort_index()
ax = low_counts.plot(kind="bar", figsize=(8, 4), title="Low count distribution, low <= 24")
ax.set_xlabel("Low numbers in draw")
ax.set_ylabel("Draw count")
plt.show()
ax = patterns["sum"].plot(kind="hist", bins=30, figsize=(10, 4), title="Draw sum distribution")
ax.set_xlabel("Sum")
plt.show()
patterns[["sum", "spread", "odd_count", "low_count", "consecutive_pairs"]].describe()
